# Prompt Golf — Minimal Training Demo

Train a Qwen3-1.7B **agent** (LoRA) to write short prompts that steer a frozen **target** LLM.
Cross-family RL on the OpenEnv Prompt Golf environment using TRL GRPO.

**Hardware**
- Recommended: L4 or A100 (Colab Pro+) — runs the headline `Qwen agent → Llama-3.2-3B target` config.
- Free T4 (16 GB): downsize the target to `Qwen/Qwen2.5-0.5B-Instruct` so everything fits.

This notebook runs a 30-step smoke training so you can verify the pipeline end-to-end on Colab in ~10 min.
For the full 500-step training that produced the demo CSVs, use HuggingFace Jobs via `training/hf_job_train.sh`.

## 1. Install dependencies

Mirrors the OpenEnv-official pin set used by HF Jobs (`pytorch/2.4.0-cuda12.4` base + uv upgrade to torch ≥ 2.8 + `trl==0.22.2`).

In [ ]:
!pip install -q -U uv
!uv pip install --system -q \
    "torch>=2.8.0" "torchvision>=0.25.0" "triton>=3.4.0" bitsandbytes \
    "transformers==4.56.2" \
    "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
    "unsloth[base] @ git+https://github.com/unslothai/unsloth"
!uv pip install --system --upgrade --no-deps -q \
    "transformers==4.56.2" tokenizers "trl==0.22.2" unsloth unsloth_zoo
!pip install -q 'openenv-core[core]>=0.2.2' 'peft>=0.13.0' 'datasets>=3.0.0' 'accelerate>=0.34.0'

## 2. Clone the env + install the package

In [ ]:
!rm -rf /content/prompt_golf_env
!git clone --depth 1 https://huggingface.co/spaces/rishabh16196/prompt_golf_env /content/prompt_golf_env
%cd /content/prompt_golf_env
!pip install -q --no-deps -e .

## 3. Log in to HuggingFace

Needed to download Qwen3-1.7B and Llama-3.2-3B-Instruct (the latter is gated — accept the license at https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct first).

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## 4. Verify the env (mock backend, CPU-only — no model load)

Quick sanity check that the env imports and resets correctly.

In [ ]:
import os
os.environ['PROMPT_GOLF_TARGET_BACKEND'] = 'mock'
os.environ['PROMPT_GOLF_JUDGE_BACKEND'] = 'mock'

from prompt_golf_env.server.prompt_golf_environment import PromptGolfEnvironment, _ALL_TASKS
from prompt_golf_env.models import GolfAction

print(f'task bank: {len(_ALL_TASKS)} tasks (20 v1 + 15 v2 + 52 tough)')

env = PromptGolfEnvironment()
obs = env.reset(task='sentiment_basic', seed=0)
print(f'\ntask: {obs.task_id}  |  budget: {obs.prompt_budget_tokens} tokens')
print(f'verbose description ({len(obs.task_description)} chars):')
print(f'  {obs.task_description[:140]}...')

# Try a hand-written prompt
result = env.step(GolfAction(prompt='Classify the sentiment as positive, negative, or neutral. Output the label only.'))
print(f'\nhand-written prompt:  reward={result.reward:+.3f}  raw={result.raw_task_score:.2f}  '
      f'tokens={result.submitted_prompt_tokens}  leak={result.leakage_penalty:.2f}')

## 5. Mini training run (30 steps)

Runs the full agent + target pipeline with a small number of steps to verify the loop works on your hardware. Defaults below are sized for **L4 (24 GB)**.

**For free T4 (16 GB)**: change `--target-model` to `Qwen/Qwen2.5-0.5B-Instruct`, drop `--num-generations 4` to `2`, and skip the judge (set `PROMPT_GOLF_JUDGE_BACKEND=mock` in the cell below).

In [ ]:
# Switch off mock backends — we want real model inference now.
del os.environ['PROMPT_GOLF_TARGET_BACKEND']
# Keep judge on mock for the smoke run unless you have an A100; the
# 8B 8-bit judge alone takes ~8 GB on top of agent + target.
os.environ['PROMPT_GOLF_JUDGE_BACKEND'] = 'mock'

In [ ]:
# `train_grpo.py` trains on ALL tasks in the bank by default. The
# --held-out-tasks flag carves out a small eval split that the GRPO
# trainer reports on each step. With max_steps=30 the loop sees a
# tiny fraction of the bank — purpose here is to verify the pipeline
# runs on your hardware, not to converge.
!python -u training/train_grpo.py \
    --agent-model Qwen/Qwen3-1.7B \
    --target-model meta-llama/Llama-3.2-3B-Instruct \
    --max-steps 30 \
    --num-generations 4 \
    --per-device-batch-size 2 \
    --gradient-accumulation-steps 2 \
    --seeds-per-task 2 \
    --learning-rate 5e-6 \
    --beta 0.04 \
    --enable-thinking \
    --max-completion-length 768 \
    --output-dir /content/outputs/grpo_demo

## 6. Inspect training metrics

In [ ]:
import json
import matplotlib.pyplot as plt

metrics_path = '/content/outputs/grpo_demo/train_metrics.jsonl'
rows = [json.loads(l) for l in open(metrics_path)]
print(f'{len(rows)} steps logged')

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
steps = [r['step'] for r in rows]
axes[0].plot(steps, [r['reward'] for r in rows], color='#1f77b4')
axes[0].axhline(0, color='gray', lw=0.5)
axes[0].set_title('reward per step'); axes[0].set_xlabel('step'); axes[0].grid(alpha=0.3)
axes[1].plot(steps, [r.get('avg_tokens', 0) for r in rows], color='#ff7f0e')
axes[1].set_title('avg prompt tokens per step'); axes[1].set_xlabel('step'); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 7. Eval the trained adapter on a few tasks

Loads the LoRA adapter you just trained and prints what it now writes for each task vs the verbose hand-written description.

In [ ]:
!python -u training/eval_before_after.py \
    --agent-model Qwen/Qwen3-1.7B \
    --adapter /content/outputs/grpo_demo/adapter_final \
    --target-model meta-llama/Llama-3.2-3B-Instruct \
    --label trained \
    --tasks tough_fallacy_classify,sentiment_basic,ner_people,format_uppercase \
    --output-json /content/outputs/eval_trained.jsonl \
    --enable-thinking

In [ ]:
rows = [json.loads(l) for l in open('/content/outputs/eval_trained.jsonl')]
for r in rows:
    print(f"\n[{r['task_id']}]  reward={r['reward']:+.3f}  raw={r['raw_task_score']:.2f}  tokens={r['tokens']}")
    print(f"  trained agent's prompt: {r['agent_prompt'][:140]!r}")

## What's next

This notebook ran 30 steps on 4 tasks — enough to verify the pipeline. The adapter checkpoints used in the demo CSVs were produced by 500-step runs over all 87 tasks, which take ~3-4h on L40S.

**To reproduce the full results:**
1. `bash training/hf_job_train.sh` — same-family Qwen→Qwen baseline (single-turn)
2. `ENABLE_THINKING=true PUSH_TO_HUB=rishabh16196/prompt-golf-qwen-to-llama bash training/hf_job_train.sh` — cross-family Qwen→Llama (the hero run)
3. `bash training/hf_job_train_multistep.sh` — multi-turn trajectory-level GRPO (warm-started from #2)
4. `bash training/hf_job_eval.sh both` — base + trained eval on either adapter
5. `python training/build_before_after_csv.py ...` — merge eval JSONLs into the demo CSV

Existing artifacts:
- Qwen→Qwen demo CSV: https://huggingface.co/rishabh16196/prompt-golf-grpo-1.5b/blob/main/evals/qwen_to_qwen_demo.csv
- Capability profiles (per task): https://huggingface.co/rishabh16196/prompt-golf-grpo-1.5b/tree/main/profiles
- Plots: https://huggingface.co/rishabh16196/prompt-golf-grpo-1.5b/tree/main/plots